# 06 - Research Experiment Harness (Intro)

A minimal grid-search harness that writes **JSONL logs**.


In [ ]:
import os, json, time
import requests


In [ ]:
BASE_URL = 'http://localhost:11434'

def run(model: str, prompt: str, temperature: float):
    payload = {'model': model, 'prompt': prompt, 'temperature': temperature, 'stream': False}
    t0 = time.time()
    r = requests.post(f'{BASE_URL}/api/generate', json=payload, timeout=300)
    r.raise_for_status()
    t1 = time.time()
    out = r.json().get('response', '')
    return {
        'ts': time.time(),
        'model': model,
        'temperature': temperature,
        'prompt': prompt,
        'latency_s': t1 - t0,
        'output_chars': len(out)
    }


In [ ]:
MODELS = ['llama3.2:3b']
TEMPS = [0.0, 0.2, 0.8]
PROMPTS = [
  'Explain RAG in a systems perspective.',
  'Give a short comparison: local inference vs cloud inference.',
  'Write pseudocode for SGD.'
]


In [ ]:
rows = []
for m in MODELS:
  for t in TEMPS:
    for p in PROMPTS:
      rows.append(run(m, p, t))

os.makedirs('../experiments/logs', exist_ok=True)
path = '../experiments/logs/experiments.jsonl'
with open(path, 'w') as f:
  for r in rows:
    f.write(json.dumps(r) + '\n')

print('Saved', path)
rows[:2]
